In [1]:
%matplotlib tk

In [2]:
import dnois
from dnois.optics import rt
import torch
from polysurf import ExtendedPolynomial


In [3]:
torch.set_default_dtype(torch.double)
torch.set_grad_enabled(False)

In [4]:
dnois.set_default('length', 'mm')
dnois.set_default('angle', 'deg')
dnois.ext.zmx.load_agf('CHINA.agf')

In [5]:
def apply_tilt_x(context, angle):
    context.phi = 90
    context.theta = -angle
    context.chi = -90

In [6]:
sq = rt.SurfaceSequence([
    rt.Plane(aperture=31.675),
    rt.Plane(aperture=24.424),
    ExtendedPolynomial(-613.98, 18.807, [
        0., 0., -2.35, 0., -2.664, 0., 0.126, 0., 0.096, 0.148, 0., 0.2840, 0.139,
    ], aperture=rt.RectangularAperture(21 * 2, 20 * 2, center_y=29), norm_radius=50, reflective=True),
    rt.Conic(-71.19, 1.645, aperture=8.2, reflective=True),
    ExtendedPolynomial(-60.58, -0.523, [
        0., 0., 6.668, 0., 6.906, 0., 0.03, 0., 0.032, 0.354, 0., 0.762, 0., 0.424,
    ], aperture=rt.RectangularAperture(22.5 * 2, 21.5 * 2, center_y=-19.9), norm_radius=50, reflective=True),
    rt.Plane(aperture=10.004),
    rt.Plane('K9', rt.RectangularAperture(8.95 * 2, 8.5 * 2)),
    rt.Plane(aperture=rt.RectangularAperture(8.95 * 2, 8.5 * 2)),
    rt.Plane(aperture=rt.RectangularAperture(4.4 * 2, 3.7 * 2)),  #IMAGE
])
sq.all_relative_()
sq[1].context.z = 80.
sq[2].context.z = 55.
sq[2].context.y = -27.67
apply_tilt_x(sq[2].context, -11.74)
sq[3].context.z = -55.
sq[3].context.y = 5.12
sq[4].context.z = 55
sq[4].context.y = 2.03 - 5.12
sq[5].context.z = -55
sq[5].context.y = -2.03 - 16.147
apply_tilt_x(sq[5].context, 1.92)
sq[6].context.z = -18.048
sq[7].context.z = -0.7
sq[8].context.z = -0.4

for idx in [3, 5, 6, 7, 8]:
    sq[idx].context.upward_in = False

In [7]:
optics = rt.CoaxialRayTracing(sq)

In [8]:
optics.plot_3d((0., 0.))